In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# 1. Dependencies and Setup

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# 2. Optional Baseline Submission

# df = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
# df.to_csv("submission.csv", index=False)

# 3. Weights & Biases Authentication

import warnings
warnings.filterwarnings('ignore')
import wandb
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))

# 4. Read Competition Data

dataset_root = '/kaggle/input/competitions/industrial-surface-defect-detection/public'
training_data = pd.read_csv(dataset_root + '/train.csv')
submission_template = pd.read_csv(dataset_root + '/sample_submission.csv')
training_files = os.listdir(dataset_root + '/train')
testing_files = os.listdir(dataset_root + '/test')

In [ ]:
# 5. Dataset Exploration

print(training_data.shape)
print(submission_template.shape)
print(len(training_files))
print(len(testing_files))

display(training_data.head())

print(training_data.columns.tolist())

print(training_data['image_id'].nunique())

print(training_data['defect_type'].unique())

display(training_data.isna().sum())

training_data['has_defect'] = training_data['mask_rle'].notna()
frequency_counts = training_data.groupby('defect_type')['has_defect'].sum()
print(frequency_counts)

frequency_counts.plot(kind='bar')
plt.title('Number of images with each defect')
plt.xlabel('Defect type')
plt.ylabel('Number of images')
plt.show()

defect_counts_by_image = training_data.groupby('image_id')['has_defect'].sum()
print(defect_counts_by_image.value_counts().sort_index())

print((defect_counts_by_image.value_counts(normalize=True).sort_index() * 100).round(2))

defect_counts_by_image.value_counts().sort_index().plot(kind='bar')
plt.title('Number of defect types per image')
plt.xlabel('Defect types in one image')
plt.ylabel('Number of images')
plt.show()

defect_indicator_table = training_data.pivot(index='image_id', columns='defect_type', values='has_defect')
print(defect_indicator_table.corr().round(2))

import random
images_folder = dataset_root + '/train/'
selected_files = random.sample(training_files, 10)
plt.figure(figsize=(16, 22))
for i in range(10):
    plt.subplot(10, 1, i + 1)
    plt.imshow(Image.open(images_folder + selected_files[i]), cmap='gray')
    plt.title(selected_files[i], fontsize=9)
    plt.axis('off')
plt.show()

In [ ]:
# 6. Run-Length Mask Conversion

def rle_decode(rle, height=128, width=800):
    mask = np.zeros(height * width, dtype=np.uint8)
    if isinstance(rle, str):
        numbers = list(map(int, rle.split()))
        starts = numbers[0::2]
        lengths = numbers[1::2]
        for s, l in zip(starts, lengths):
            mask[s - 1:s - 1 + l] = 1
    return mask.reshape(width, height).T

def rle_encode(mask):
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    changes = np.where(pixels[1:] != pixels[:-1])[0] + 1
    changes[1::2] -= changes[::2]
    return ' '.join(map(str, changes))

row = training_data[training_data.mask_rle.notna()].iloc[0]
mask = rle_decode(row.mask_rle)
back = rle_encode(mask)
print(back == row.mask_rle)

In [ ]:
# 8. Inspecting Ground-Truth Defects

training_data['has_defect'] = training_data['mask_rle'].notna()
images_folder = dataset_root + '/train/'
sampled_rows = training_data[training_data.has_defect].sample(6)
plt.figure(figsize=(16, 15))
for i, (_, r) in enumerate(sampled_rows.iterrows()):
    img = Image.open(images_folder + r.image_id + '.jpg')
    mask = rle_decode(r.mask_rle)
    plt.subplot(6, 1, i + 1)
    plt.imshow(img, cmap='gray')
    plt.imshow(np.ma.masked_equal(mask, 0), alpha=0.6, cmap='Reds', vmin=0, vmax=1)
    plt.title(r.image_id + ' | ' + r.defect_type, fontsize=9)
    plt.axis('off')
plt.show()

In [ ]:
# 9. Creating Training and Validation Partitions
# We split the images 80/20. The split is stratified by defect combination,
# so rare defects like defect_b appear in both parts.

from sklearn.model_selection import train_test_split
defect_indicator_table = training_data.pivot(index='image_id', columns='defect_type', values='has_defect')
defect_patterns = defect_indicator_table.apply(lambda r: ''.join((c[-1] for c in defect_indicator_table.columns if r[c])) or 'clean', axis=1)
frequency_counts = defect_patterns.value_counts()
defect_patterns = defect_patterns.where(defect_patterns.map(frequency_counts) >= 10, 'other')
training_ids, validation_ids = train_test_split(defect_indicator_table.index, test_size=0.2, stratify=defect_patterns, random_state=42)
print('train images:', len(training_ids))
print('val images:', len(validation_ids))

print((defect_indicator_table.loc[training_ids].mean() * 100).round(2))
print((defect_indicator_table.loc[validation_ids].mean() * 100).round(2))

In [ ]:
# 10. Dataset Pipeline and Batch Loading

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
defect_labels = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
encoded_masks_table = training_data.pivot(index='image_id', columns='defect_type', values='mask_rle')

class DefectDataset(Dataset):

    def __init__(self, image_ids, augment=False):
        self.image_ids = list(image_ids)
        self.augment = augment

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, i):
        image_id = self.image_ids[i]
        img = Image.open(dataset_root + '/train/' + image_id + '.jpg').convert('RGB')
        img = np.array(img) / 255.0
        masks = []
        for d in defect_labels:
            masks.append(rle_decode(encoded_masks_table.loc[image_id, d]))
        masks = np.stack(masks, axis=-1)
        if self.augment and np.random.rand() < 0.5:
            img = np.fliplr(img)
            masks = np.fliplr(masks)
        if self.augment and np.random.rand() < 0.5:
            img = np.flipud(img)
            masks = np.flipud(masks)
        if self.augment:
            img = img * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.08, 0.08)
            img = np.clip(img, 0, 1)
        img = torch.tensor(img.copy(), dtype=torch.float32).permute(2, 0, 1)
        masks = torch.tensor(masks.copy(), dtype=torch.float32).permute(2, 0, 1)
        return (img, masks)
training_dataset = DefectDataset(training_ids, augment=True)
validation_dataset = DefectDataset(validation_ids)
training_flags = defect_indicator_table.loc[training_ids]
class_frequency = training_flags[defect_labels].mean()
w = np.ones(len(training_flags))
for d in defect_labels:
    w += training_flags[d].values / np.sqrt(class_frequency[d])
weighted_sampler = WeightedRandomSampler(w, num_samples=len(w), replacement=True)
training_loader = DataLoader(training_dataset, batch_size=16, sampler=weighted_sampler, num_workers=2)
validation_loader = DataLoader(validation_dataset, batch_size=16, shuffle=False, num_workers=2)
x, y = training_dataset[0]
print(x.shape, y.shape)

!pip install -q segmentation-models-pytorch

In [ ]:
# 11. Segmentation Network and Encoder

import segmentation_models_pytorch as smp
device = 'cuda'
model = smp.Unet(encoder_name='efficientnet-b3', encoder_weights='imagenet', in_channels=3, classes=4).to(device)
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

x, y = next(iter(training_loader))
x = x.to(device)
out = model((x - mean) / std)
print(out.shape)

In [ ]:
# 12. Objective and Validation Statistics

positive_weights = torch.tensor([2.0, 3.0, 1.0, 1.5], device=device).view(1, 4, 1, 1)
bce_loss = torch.nn.BCEWithLogitsLoss(pos_weight=positive_weights)
dice_loss = smp.losses.DiceLoss(mode='multilabel', from_logits=True)

def loss_fn(out, y):
    return bce_loss(out, y) + dice_loss(out, y)

def batch_stats(pred, gt):
    p, g = (pred.flatten(2), gt.flatten(2))
    psum, ground_truth_areas = (p.sum(2), g.sum(2))
    has = ground_truth_areas > 0
    d = 2 * (p * g).sum(2) / (psum + ground_truth_areas + 1e-09)
    fp = 1 - psum / p.shape[2]
    empty = (psum == 0).float()
    return (d[has].tolist(), fp[~has].tolist(), empty[~has].tolist())

# 13. Optimizer, Schedule, and Logging

NUM_EPOCHS = 30
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-06)
run = wandb.init(project='21f1001536-t32026', name='v2-unet-effb3', mode='online', tags=['unet', 'efficientnet-b3', 'sampler', 'pos_weight', 'cosine', 'tta'], notes='v2b: U-Net EfficientNet-B3 + WeightedRandomSampler + pos_weight + brightness aug + cosine LR + TTA + per-class thresholds', config={'lr': 0.0003, 'epochs': NUM_EPOCHS, 'batch_size': 16, 'model': 'unet-efficientnet-b3', 'sampler': True, 'pos_weight': '2,3,1,1.5', 'scheduler': 'cosine'})

In [ ]:
# 14. Fitting and Model Selection

import time
best = 0
top_epoch = 0
start = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    model.train()
    average_train_loss = 0
    for x, y in training_loader:
        x, y = (x.to(device), y.to(device))
        optimizer.zero_grad()
        out = model((x - mean) / std)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        average_train_loss += loss.item()
    average_train_loss /= len(training_loader)
    model.eval()
    D, F, E = ([], [], [])
    with torch.no_grad():
        for x, y in validation_loader:
            x, y = (x.to(device), y.to(device))
            pred = (torch.sigmoid(model((x - mean) / std)) > 0.5).float()
            d, f, e = batch_stats(pred, y)
            D += d
            F += f
            E += e
    dice_s, fp_s, empty_s = (np.mean(D), np.mean(F), np.mean(E))
    score = 0.7 * dice_s + 0.2 * fp_s + 0.1 * empty_s
    if score > best:
        best = score
        top_epoch = epoch
        torch.save(model.state_dict(), 'unet_effb3_v2b.pth')
    scheduler.step()
    elapsed_epoch_seconds = time.time() - t0
    elapsed_total_minutes = (time.time() - start) / 60
    wandb.log({'epoch': epoch, 'train_loss': average_train_loss, 'val_dice': dice_s, 'val_fp': fp_s, 'val_empty': empty_s, 'comp_metric': score, 'epoch_time_sec': elapsed_epoch_seconds, 'lr': optimizer.param_groups[0]['lr']})
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | loss {average_train_loss:.4f} | dice {dice_s:.3f} | fp {fp_s:.4f} | empty {empty_s:.3f} | score {score:.3f} | best {best:.3f} (ep {top_epoch}) | {elapsed_epoch_seconds:.0f}s | total {elapsed_total_minutes:.1f} min')
wandb.summary['best_score'] = best
wandb.summary['best_epoch'] = top_epoch
wandb.finish()

import numpy as np
import torch
CHECKPOINT_FILE = 'unet_effb3_v2b.pth'
model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=device))
model.eval()

def predict_tta(x):
    """Average of normal + horizontal flip + vertical flip predictions."""
    with torch.no_grad():
        p = torch.sigmoid(model((x - mean) / std))
        p += torch.sigmoid(model((x.flip(3) - mean) / std)).flip(3)
        p += torch.sigmoid(model((x.flip(2) - mean) / std)).flip(2)
    return (p / 3).cpu().numpy().astype(np.float16)
validation_probabilities, validation_masks = ([], [])
for x, y in validation_loader:
    validation_probabilities.append(predict_tta(x.to(device)))
    validation_masks.append(y.numpy() > 0.5)
validation_probabilities = np.concatenate(validation_probabilities)
validation_masks = np.concatenate(validation_masks)
print('val_probs:', validation_probabilities.shape, 'val_masks:', validation_masks.shape)
ground_truth_areas = validation_masks.sum((2, 3))
has = ground_truth_areas > 0
positive_total, negative_total, P = (max(has.sum(), 1), max((~has).sum(), 1), 128 * 800)
THRESHOLD_OPTIONS = [0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6]
MIN_AREA_OPTIONS = [0, 50, 100, 200, 400, 800]
optimal_parameters, total = ({}, 0)
for k, name in enumerate(defect_labels):
    pk, gk, hk, gs = (validation_probabilities[:, k], validation_masks[:, k], has[:, k], ground_truth_areas[:, k])
    best = (-1, None, None)
    best_dice = 0
    for t in THRESHOLD_OPTIONS:
        pred = pk > t
        ps0 = pred.sum((1, 2))
        it0 = (pred & gk).sum((1, 2))
        for a in MIN_AREA_OPTIONS:
            keep = ps0 >= a
            ps, it = (np.where(keep, ps0, 0), np.where(keep, it0, 0))
            dice_k = 2 * it / (ps + gs + 1e-09)
            c = 0.7 * dice_k[hk].sum() / positive_total + 0.2 * (1 - ps[~hk] / P).sum() / negative_total + 0.1 * (ps[~hk] == 0).sum() / negative_total
            if c > best[0]:
                best = (c, t, a)
                best_dice = dice_k[hk].mean()
    optimal_parameters[name] = (best[1], best[2])
    total += best[0]
    print(name, '| thr', best[1], '| min_area', best[2], '| dice on defect images:', round(float(best_dice), 3))
print('\nVal score (TTA + per-class thresholds):', round(float(total), 4))
print('best_params =', optimal_parameters)

In [ ]:
# 15. Test-Set Inference and Mask Encoding

testing_ids = submission_template['image_id'].unique()
predicted_rles = {}
for i in range(0, len(testing_ids), 16):
    ids = testing_ids[i:i + 16]
    imgs = [np.array(Image.open(dataset_root + '/test/' + im + '.jpg').convert('RGB')) / 255.0 for im in ids]
    x = torch.tensor(np.stack(imgs), dtype=torch.float32).permute(0, 3, 1, 2).to(device)
    prob = predict_tta(x).astype(np.float32)
    for j, image_id in enumerate(ids):
        for k, d in enumerate(defect_labels):
            thr, area = optimal_parameters[d]
            mask = (prob[j, k] > thr).astype(np.uint8)
            if mask.sum() < area:
                mask[:] = 0
            predicted_rles[image_id, d] = rle_encode(mask) if mask.sum() > 0 else ' '
print('test predictions done:', len(predicted_rles))

# 16. Prepare and Verify Submission

import csv
final_submission = submission_template.copy()
final_submission['mask_rle'] = [predicted_rles[i, d] for i, d in zip(final_submission['image_id'], final_submission['defect_type'])]
final_submission.to_csv('submission.csv', index=False, quoting=csv.QUOTE_ALL)
submission_check = pd.read_csv('submission.csv', keep_default_na=False, dtype=str)
reference_submission = pd.read_csv(dataset_root + '/sample_submission.csv', keep_default_na=False, dtype=str)
print('shape:', submission_check.shape, '| same columns:', list(submission_check.columns) == list(reference_submission.columns))
print('ID/image_id/defect_type match:', all(((submission_check[c] == reference_submission[c]).all() for c in ['ID', 'image_id', 'defect_type'])))
print('Empty predictions:', (submission_check['mask_rle'] == ' ').mean())
print(submission_check.head(8))